In [2]:
pip install numpy pandas matplotlib seaborn tqdm pillow torch torchvision scikit-learn

In [3]:
import os
import time
import copy
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')

# Set device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")


In [ ]:

DATASET_PATH = "C:\\Users\\91928\\OneDrive\\Desktop\\Comparative Analysis\\dataset" # Change this to your 
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

def get_image_paths_and_labels(dataset_path):
    image_paths = []
    labels = []
    
    if not os.path.exists(dataset_path):
        print(f"Warning: The dataset path '{dataset_path}' does not exist! Please update DATASET_PATH.")
        return [], [], []
        
    class_names = sorted(os.listdir(dataset_path))
    class_names = [c for c in class_names if os.path.isdir(os.path.join(dataset_path, c))]
    
    class_to_idx = {class_name: i for i, class_name in enumerate(class_names)}
    
    for class_name in class_names:
        class_dir = os.path.join(dataset_path, class_name)
        for img_name in os.listdir(class_dir):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                image_paths.append(os.path.join(class_dir, img_name))
                labels.append(class_to_idx[class_name])
                
    return np.array(image_paths), np.array(labels), class_names

# Get paths and labels
image_paths, labels, class_names = get_image_paths_and_labels(DATASET_PATH)

if len(class_names) > 0:
    print(f"Detected {len(class_names)} classes: {class_names}")
    print(f"Total Images: {len(image_paths)}")


EDA

In [ ]:
if len(class_names) > 0:
    # Class Distribution Plot
    unique_labels, counts = np.unique(labels, return_counts=True)
    plt.figure(figsize=(8, 5))
    sns.barplot(x=[class_names[i] for i in unique_labels], y=counts, palette='viridis')
    plt.title('Class Distribution in Dataset')
    plt.xlabel('Class')
    plt.ylabel('Number of Images')
    plt.show()

    # Sample Visualization
    fig, axes = plt.subplots(1, len(class_names), figsize=(15, 5))
    for i, class_name in enumerate(class_names):
        # find the first image of this class
        idx = np.where(labels == i)[0][0]
        img_path = image_paths[idx]
        try:
            img = Image.open(img_path).convert('RGB')
            axes[i].imshow(img)
            axes[i].set_title(class_name)
            axes[i].axis('off')
        except Exception as e:
            print(f"Could not load image {img_path}: {e}")
    plt.tight_layout()
    plt.show()


Data Preprocessing



In [ ]:
# Train-Val-Test Split
if len(image_paths) > 0:
    X_train_val, X_test, y_train_val, y_test = train_test_split(image_paths, labels, test_size=0.15, stratify=labels, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.15/0.85, stratify=y_train_val, random_state=42)

    print(f"Training Set: {len(X_train)}")
    print(f"Validation Set: {len(X_val)}")
    print(f"Testing Set: {len(X_test)}")
    
# Image Transforms
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Custom Dataset Class
class MedicalImageDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.file_paths)
        
    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.long)

if len(image_paths) > 0:
    train_dataset = MedicalImageDataset(X_train, y_train, transform=train_transforms)
    val_dataset = MedicalImageDataset(X_val, y_val, transform=test_val_transforms)
    test_dataset = MedicalImageDataset(X_test, y_test, transform=test_val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


In [ ]:
# Global Dictionary to keep track of model evaluations
evaluation_results = []

def train_model(model, dataloaders, criterion, optimizer, num_epochs=10, patience=3):
    """
    Generalized training loop featuring early stopping and best-model checkpointing.
    """
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                dataloader = dataloaders['train']
            else:
                model.eval()
                dataloader = dataloaders['val']

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data.
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.double() / len(dataloader.dataset)
            
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            print(f"{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # deep copy the model if it has better validation accuracy
            if phase == 'val':
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
        
        if epochs_no_improve >= patience:
            print("Early stopping triggered!")
            break

        print()

    time_elapsed = time.time() - since
    print(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
    print(f"Best Val Acc: {best_acc:4f}")

    model.load_state_dict(best_model_wts)
    return model, history, time_elapsed

def evaluate_model(model_name, model, dataloader, training_time, is_binary=False):
    """
    Evaluates the model on test set and computes precision, recall, f1, and roc auc.
    """
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            if is_binary:
                all_probs.extend(probs[:, 1].cpu().numpy())
            else:
                all_probs.extend(probs.cpu().numpy())
                
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    try:
        if is_binary:
            roc_auc = roc_auc_score(all_labels, all_probs)
        else:
            roc_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    except ValueError:
        roc_auc = 0.0 # Handles edge cases where one class doesn't appear
        
    evaluation_results.append({
        'Model': model_name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': roc_auc,
        'Training Time (s)': training_time
    })
    
    print(f"Metrics for {model_name}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}, AUC={roc_auc:.4f}")
    
    # Plot Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} Confusion Matrix')
    plt.ylabel('True Class')
    plt.xlabel('Predicted Class')
    plt.show()


CNN


In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # 224 / 2 / 2 / 2 = 28 -> 128 features 28x28
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(-1, 128 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

num_classes = len(class_names) if 'class_names' in locals() else 2
is_binary = True if num_classes == 2 else False

# To avoid long executions during structure verification, we will set mock variables 
# if dataset is empty. But if fully populated, it runs.
if 'train_loader' in locals():
    custom_cnn = CustomCNN(num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(custom_cnn.parameters(), lr=1e-4)

    dataloaders = {'train': train_loader, 'val': val_loader}
    
    print("Training Custom CNN...")
    custom_cnn, history_cnn, time_cnn = train_model(custom_cnn, dataloaders, criterion, optimizer, num_epochs=15, patience=4)
    evaluate_model("Custom CNN", custom_cnn, test_loader, time_cnn, is_binary=is_binary)


Transfer Learning Models(RestNet50, DenseNet121, EfficientNetB0)


In [ ]:
def get_pretrained_model(model_name, num_classes):
    if model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, num_classes)
    elif model_name == 'densenet121':
        model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        num_ftrs = model.classifier.in_features
        model.classifier = nn.Linear(num_ftrs, num_classes)
    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        num_ftrs = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_ftrs, num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")
        
    return model.to(device)

if 'train_loader' in locals():
    # Model 1: ResNet50
    resnet_model = get_pretrained_model('resnet50', num_classes)
    optimizer_ft = optim.Adam(resnet_model.fc.parameters(), lr=1e-3) # Only optimized the head first
    print("\nTraining ResNet50...")
    resnet_model, _, time_res = train_model(resnet_model, dataloaders, criterion, optimizer_ft, num_epochs=10)
    evaluate_model("ResNet50", resnet_model, test_loader, time_res, is_binary=is_binary)

    # Model 2: DenseNet121
    dense_model = get_pretrained_model('densenet121', num_classes)
    optimizer_ft = optim.Adam(dense_model.classifier.parameters(), lr=1e-3)
    print("\nTraining DenseNet121...")
    dense_model, _, time_dense = train_model(dense_model, dataloaders, criterion, optimizer_ft, num_epochs=10)
    evaluate_model("DenseNet121", dense_model, test_loader, time_dense, is_binary=is_binary)
    
    # Model 3: EfficientNetB0
    eff_model = get_pretrained_model('efficientnet_b0', num_classes)
    optimizer_ft = optim.Adam(eff_model.classifier[1].parameters(), lr=1e-3)
    print("\nTraining EfficientNetB0...")
    eff_model, _, time_eff = train_model(eff_model, dataloaders, criterion, optimizer_ft, num_epochs=10)
    evaluate_model("EfficientNetB0", eff_model, test_loader, time_eff, is_binary=is_binary)


Meta Learning / Few-Shot Learning (Prototypical Networks)



In [ ]:
# Prototypical Network implementation
class PrototypicalEncoder(nn.Module):
    def __init__(self):
        super(PrototypicalEncoder, self).__init__()
        # Use a frozen pre-trained feature extractor for speed/stability in this notebook
        resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
    def forward(self, x):
        x = self.features(x)
        return x.view(x.size(0), -1)
        
def compute_prototypes(support_embeddings, support_labels, classes):
    """ Computes the centroid (prototype) of each class in the embedding space."""
    prototypes = []
    for c in classes:
        mask = (support_labels == c)
        class_embeddings = support_embeddings[mask]
        class_prototype = class_embeddings.mean(dim=0)
        prototypes.append(class_prototype)
    return torch.stack(prototypes)

if 'train_loader' in locals():
    # Initialize ProtoNet
    proto_net = PrototypicalEncoder().to(device)
    proto_optimizer = optim.Adam(proto_net.parameters(), lr=1e-4)

    # Simplified Episodic Training for demonstration
    # In a full setup, you use Episodic Samplers targeting N-way K-shot scenarios.
    
    proto_epochs = 10
    print("\nTraining Prototypical Network (Simplified Episodic)")
    start_proto = time.time()
    
    for epoch in range(proto_epochs):
        proto_net.train()
        total_loss = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Split batch into Support Set and Query Set for few-shot task emulation
            split_idx = len(inputs) // 2
            support_x, query_x = inputs[:split_idx], inputs[split_idx:]
            support_y, query_y = labels[:split_idx], labels[split_idx:]
            
            proto_optimizer.zero_grad()
            
            embeddings = proto_net(inputs)
            support_embeddings = embeddings[:split_idx]
            query_embeddings = embeddings[split_idx:]
            
            unique_classes = torch.unique(support_y)
            if len(unique_classes) < num_classes:
                continue # Skip batch if not all classes are represented in support set for simple demo
            
            prototypes = compute_prototypes(support_embeddings, support_y, unique_classes)
            
            # Distance from query samples to prototypes
            # dist -> shape: [num_queries, num_classes]
            dist = torch.cdist(query_embeddings, prototypes)
            
            # Label mapping relative to unique_classes
            class_to_idx = {c.item(): i for i, c in enumerate(unique_classes)}
            mapped_query_y = torch.tensor([class_to_idx[y.item()] for y in query_y]).to(device)

            log_p_y = F.log_softmax(-dist, dim=1) # softmax over negative distance
            loss = F.nll_loss(log_p_y, mapped_query_y)
            
            loss.backward()
            proto_optimizer.step()
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1} ProtoNet Loss: {total_loss/len(train_loader):.4f}")
        
    time_proto = time.time() - start_proto
    
    # Evaluate ProtoNet
    proto_net.eval()
    all_preds_proto = []
    all_labels_proto = []
    
    print("Evaluating ProtoNet on Test Set...")
    # Get a fixed support set from training data
    support_inputs, support_labels = next(iter(train_loader))
    support_inputs, support_labels = support_inputs.to(device), support_labels.to(device)
    unique_classes_test = torch.unique(support_labels)
    
    with torch.no_grad():
        support_embeddings = proto_net(support_inputs)
        prototypes = compute_prototypes(support_embeddings, support_labels, unique_classes_test)
        
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            query_embeddings = proto_net(inputs)
            dist = torch.cdist(query_embeddings, prototypes)
            
            _, mapped_preds = torch.min(dist, 1) # min distance
            # Map back to original labels
            for p, l in zip(mapped_preds, labels):
                actual_pred_label = unique_classes_test[p].item()
                all_preds_proto.append(actual_pred_label)
                all_labels_proto.append(l.item())
                
    acc_proto = accuracy_score(all_labels_proto, all_preds_proto)
    prec_proto = precision_score(all_labels_proto, all_preds_proto, average='macro', zero_division=0)
    rec_proto = recall_score(all_labels_proto, all_preds_proto, average='macro', zero_division=0)
    f1_proto = f1_score(all_labels_proto, all_preds_proto, average='macro', zero_division=0)
    
    evaluation_results.append({
        'Model': 'Prototypical Net', 'Accuracy': acc_proto, 'Precision': prec_proto, 
        'Recall': rec_proto, 'F1 Score': f1_proto, 'ROC-AUC': 0, 'Training Time (s)': time_proto
    })
    
    print(f"Metrics for ProtoNet: Acc={acc_proto:.4f}, F1={f1_proto:.4f}")


Model Comparison



In [ ]:
if evaluation_results:
    results_df = pd.DataFrame(evaluation_results)
    display(results_df)

    # Plot Accuracy and F1 Scores
    results_df_melted = results_df.melt(id_vars='Model', value_vars=['Accuracy', 'F1 Score'], var_name='Metric', value_name='Score')
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=results_df_melted, x='Model', y='Score', hue='Metric', palette='magma')
    plt.title('Model Comparison: Accuracy & F1-Score')
    plt.ylim(0, 1.1)
    plt.ylabel('Score')
    plt.xticks(rotation=45)
    plt.legend(loc='lower right')
    plt.show()


Model Selection



In [ ]:
if evaluation_results:
    best_row_idx = results_df['F1 Score'].idxmax()
    best_model_name = results_df.loc[best_row_idx, 'Model']
    best_f1 = results_df.loc[best_row_idx, 'F1 Score']
    
    print(f"🏆 Best Model: {best_model_name} with F1-Score of {best_f1:.4f}")
    
    # Save Model Weights Logic (pseudo-implementation)
    print(f"Saving {best_model_name} weights to '{best_model_name.replace(' ', '_')}_best.pth'")
    # torch.save(resnet_model.state_dict(), 'best_model.pth')


Single Image Prediction

In [ ]:
def predict_image(image_path, model, class_names):
    if not os.path.exists(image_path):
        print("Image not found...")
        return
        
    img = Image.open(image_path).convert('RGB')
    input_tensor = test_val_transforms(img).unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        conf, pred = torch.max(probs, 1)
        
    plt.imshow(img)
    plt.title(f"Prediction: {class_names[pred.item()]} (Conf: {conf.item():.4f})")
    plt.axis('off')
    plt.show()

# Example usage (Uncomment and put an actual image path)
# predict_image('test_image.jpg', resnet_model, class_names)
